# Olist Brazilian E-Commerce — Task 1

## Load the Olist dataset into PostgreSQL with Docker

This notebook implements Task 1: download/prepare the Olist CSV files, connect to PostgreSQL running in Docker, load the tables, validate the data, test joins, and create the first delivery-lateness target.

Dataset: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


## 1. Prerequisites

Install Docker Desktop, Python 3.10+, Jupyter/VS Code, and have access to the Kaggle dataset.

Expected CSV files: `olist_customers_dataset.csv`, `olist_orders_dataset.csv`, `olist_products_dataset.csv`, `olist_sellers_dataset.csv`, `olist_order_items_dataset.csv`, `olist_order_payments_dataset.csv`, `olist_order_reviews_dataset.csv`, `olist_geolocation_dataset.csv`, `product_category_name_translation.csv`.

## 2. Install Python packages

In [29]:
%pip install pandas psycopg[binary] sqlalchemy python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\ahmed\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 3. Create the project folders

Run once. CSV files will be stored under `data/raw/`.

In [30]:
from pathlib import Path
ROOT=Path.cwd()
DATA_DIR=ROOT/'data'/'raw'
DATA_DIR.mkdir(parents=True,exist_ok=True)
print(ROOT, DATA_DIR)

g:\course\MLOPS\Olist_Task1_Notebook_Solution\olist_task1_notebook g:\course\MLOPS\Olist_Task1_Notebook_Solution\olist_task1_notebook\data\raw


## 4. Download the Kaggle data

Download and extract the dataset into `data/raw/`, or from a terminal run:

```bash
kaggle datasets download -d olistbr/brazilian-ecommerce -p data/raw --unzip
```

In [31]:
expected=['olist_customers_dataset.csv','olist_orders_dataset.csv','olist_products_dataset.csv','olist_sellers_dataset.csv','olist_order_items_dataset.csv','olist_order_payments_dataset.csv','olist_order_reviews_dataset.csv','olist_geolocation_dataset.csv','product_category_name_translation.csv']
for f in expected: print(('OK   ' if (DATA_DIR/f).exists() else 'MISS ')+f)

OK   olist_customers_dataset.csv
OK   olist_orders_dataset.csv
OK   olist_products_dataset.csv
OK   olist_sellers_dataset.csv
OK   olist_order_items_dataset.csv
OK   olist_order_payments_dataset.csv
OK   olist_order_reviews_dataset.csv
OK   olist_geolocation_dataset.csv
OK   product_category_name_translation.csv


## 5. Start PostgreSQL with Docker

Create `docker-compose.yml` in the project root:

```yaml
services:
  postgres:
    image: postgres:16
    container_name: olist-postgres
    restart: unless-stopped
    environment:
      POSTGRES_DB: olist
      POSTGRES_USER: olist
      POSTGRES_PASSWORD: olist
    ports:
      - "5432:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data
volumes:
  postgres_data:
```

Then run: `docker compose up -d` and `docker compose ps`.

In [32]:
%pip install pandas sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\ahmed\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 6. Create the PostgreSQL schema

In [33]:
from sqlalchemy import create_engine, text

MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3307
MYSQL_USER = "root"
MYSQL_PASSWORD = "1234"
DATABASE = "olist"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{DATABASE}?charset=utf8mb4",
    pool_pre_ping=True
)

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DATABASE()")).scalar())

Connected to: olist


## 7. Load the CSV files with PostgreSQL COPY

Load parent tables before child tables because of foreign keys. `COPY` is used for efficient bulk loading.

In [34]:
DATA_DIR = Path.cwd() / "data" / "raw"

print(DATA_DIR)

g:\course\MLOPS\Olist_Task1_Notebook_Solution\olist_task1_notebook\data\raw


In [35]:
for file in DATA_DIR.glob("*.csv"):
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [36]:
table_mapping = {
    "olist_customers_dataset.csv": "customers",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_geolocation_dataset.csv": "geolocation",
    "product_category_name_translation.csv": "product_category_name_translation"
}

In [37]:
load_order = [
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv"
]

In [38]:
import pandas as pd

for filename in load_order:

    file_path = DATA_DIR / filename
    table_name = table_mapping[filename]

    print(f"\n{'='*60}")
    print(f"Loading: {filename}")
    print(f"Table:   {table_name}")
    print(f"{'='*60}")

    total_rows = 0

    for chunk in pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        chunksize=10_000
    ):

        chunk.to_sql(
            table_name,
            con=engine,
            if_exists="append",
            index=False,
            chunksize=1000,
            method="multi"
        )

        total_rows += len(chunk)

        print(f"Loaded: {total_rows:,} rows", end="\r")

    print(f"\n✓ Finished {table_name}: {total_rows:,} rows")


Loading: olist_customers_dataset.csv
Table:   customers


DatabaseError: Execution failed on sql 'INSERT INTO customers (customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state) VALUES (:customer_id_m0, :customer_unique_id_m0, :customer_zip_code_prefix_m0, :customer_city_m0, :customer_state_m0), (:customer_id_m1, :customer_unique_id_m1, :customer_zip_code_prefix_m1, :customer_city_m1, :customer_state_m1), (:customer_id_m2, :customer_unique_id_m2, :customer_zip_code_prefix_m2, :customer_city_m2, :customer_state_m2), (:customer_id_m3, :customer_unique_id_m3, :customer_zip_code_prefix_m3, :customer_city_m3, :customer_state_m3), (:customer_id_m4, :customer_unique_id_m4, :customer_zip_code_prefix_m4, :customer_city_m4, :customer_state_m4), (:customer_id_m5, :customer_unique_id_m5, :customer_zip_code_prefix_m5, :customer_city_m5, :customer_state_m5), (:customer_id_m6, :customer_unique_id_m6, :customer_zip_code_prefix_m6, :customer_city_m6, :customer_state_m6), (:customer_id_m7, :customer_unique_id_m7, :customer_zip_

## 8. Validate row counts

In [45]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM customers

UNION ALL

SELECT 'orders', COUNT(*)
FROM orders

UNION ALL

SELECT 'products', COUNT(*)
FROM products

UNION ALL

SELECT 'sellers', COUNT(*)
FROM sellers

UNION ALL

SELECT 'order_items', COUNT(*)
FROM order_items

UNION ALL

SELECT 'order_payments', COUNT(*)
FROM order_payments

UNION ALL

SELECT 'order_reviews', COUNT(*)
FROM order_reviews

UNION ALL

SELECT 'geolocation', COUNT(*)
FROM geolocation

UNION ALL

SELECT 'product_category_name_translation', COUNT(*)
FROM product_category_name_translation

ORDER BY table_name;
"""

df_counts = pd.read_sql(query, engine)

display(df_counts)

,table_name,row_count
0,customers,99441
1,geolocation,1000163
2,orders,99441
3,order_items,112650
4,order_payments,103886
5,order_reviews,99224
6,products,32951
7,product_category_name_translation,71
8,sellers,3095


In [40]:
import os

data_path = "data/raw"

geolocation = pd.read_csv(
    os.path.join(data_path, "olist_geolocation_dataset.csv")
)

order_reviews = pd.read_csv(
    os.path.join(data_path, "olist_order_reviews_dataset.csv")
)

product_category_translation = pd.read_csv(
    os.path.join(data_path, "product_category_name_translation.csv")
)

In [42]:
print("Geolocation:", geolocation.shape)
print("Order Reviews:", order_reviews.shape)
print("Category Translation:", product_category_translation.shape)

Geolocation: (1000163, 5)
Order Reviews: (99224, 7)
Category Translation: (71, 2)


In [44]:
geolocation.to_sql(
    "geolocation",
    engine,
    if_exists="replace",
    index=False
)

order_reviews.to_sql(
    "order_reviews",
    engine,
    if_exists="replace",
    index=False
)

product_category_translation.to_sql(
    "product_category_name_translation",
    engine,
    if_exists="replace",
    index=False
)

71